# Unit 2 / Chapter 2: Quantum Algorithms that are Relevant to AI

> **Main Learning Objective:** Understand the four quantum algorithms most relevant to AI: quantum parallelism, Grover's search, the Quantum Fourier Transform, and quantum sampling. By the end of this unit, you will know when each one applies and have hands-on experience running each of them on a small example.

| Section | Topic | Why it matters for AI |
|---|---|---|
| 2.1 | Quantum Parallelism | Exploring exponential search spaces in one shot |
| 2.2 | Grover's Search | Quadratic speedup for unstructured search |
| 2.3 | Quantum Fourier Transform | Subroutine behind many quantum ML routines |
| 2.4 | Sampling and Probability | Powering generative models, Bayesian inference, MCMC |

By the end of this unit you should be able to:

1. Explain quantum parallelism in plain language.
2. Walk through one Grover iteration on a small example.
3. Describe what the QFT does to a quantum state, and why it's useful as a subroutine.
4. Recognize when an AI problem is essentially a sampling problem.
5. Identify which of these algorithms could plausibly help with a given AI task.

---

# Unit 2: Quantum Algorithms that are Relevant to AI

## Setup:

Run the cell below once to install the libraries you'll need. We're using small, well-known packages so the notebook works on a vanilla Python install.

In [ ]:
# Run this once, installs run silently if already present.
import sys, subprocess
for pkg in ["numpy", "matplotlib", "scipy", "ipywidgets"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])
print("All libraries ready.")

---
## Course check-in

This logs that you started **Unit 2**. You will be asked to enter the email you signed up with so we can track your progress and email your certificate when you finish all units.

In [ ]:
# ============================================================
# COURSE TRACKING - do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 2
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

In [ ]:
# Standard imports we will reuse throughout the notebook.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, patches
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, Image, display, Markdown
import math, cmath, random
from itertools import product

# Make the random number generators reproducible.
np.random.seed(42)
random.seed(42)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
print("Imports done.")

---
# Section 2.1-Quantum Parallelism:


A classical computer with **n bits** holds **exactly one** of the 2ⁿ possible bit-strings at any moment. A quantum computer with **n qubits** can hold a *superposition* of **all 2ⁿ** bit-strings at the same time. When you apply a function to that superposition, you are evaluating the function on every input at once.

This is called **quantum parallelism**, and it sits at the heart of every quantum speed-up.

> **Important caveat.** Quantum parallelism does **not** mean you get all 2ⁿ outputs for free. Measurement collapses the state to a single answer. The art of quantum algorithm design is *interfering* the parallel computations so the right answer becomes the most likely one when you measure.

## Classical vs. Quantum parallelism

| | **Classical parallelism** | **Quantum parallelism** |
|---|---|---|
| What stores the data? | Many separate bits / cores | A single superposition state |
| How does it scale? | Linear in number of cores | Exponential in number of qubits (2ⁿ) |
| Cost of "more parallel work" | Add more hardware | Add one more qubit |
| What you can read out | Every parallel answer | One sample from a distribution |
| Where the speed-up comes from | Doing more work at once | Interference + measurement design |

## Why AI cares

Almost every hard problem in modern AI is a search over an enormous space:

- **Hyperparameter search**: finding the best learning rate, depth, dropout, etc. for a neural network.
- **Feature selection**: choosing which of *m* features to keep is a search over 2ᵐ subsets.
- **Combinatorial optimization**: vehicle routing, scheduling, graph problems.
- **Reinforcement learning policies**: the space of strategies in a complex environment.

If a quantum computer can *touch* every option in a single state, even a polynomial speed-up over the classical search is enormous when *m* gets large.

### Visualizing exponential blow-up

The plot below shows how the size of the search space grows. Notice the y-axis is log-scaled- every added qubit *doubles* the space.

In [ ]:
qubits = np.arange(1, 31)
states = 2.0 ** qubits

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(qubits, states, marker='o', color='#5B2C91', linewidth=2)
ax.fill_between(qubits, 1, states, alpha=0.15, color='#5B2C91')
ax.set_xlabel("Number of qubits  n")
ax.set_ylabel("States that can be in superposition  (2ⁿ)")
ax.set_title("One more qubit = double the search space")
ax.grid(True, which='both', alpha=0.3)

# Call out a few sizes that matter for AI.
for n, label in [(10, "1,024 hyper-param combos"),
                 (20, "≈ 1M feature subsets"),
                 (30, "≈ 1B states")]:
    ax.annotate(label, xy=(n, 2**n), xytext=(n-7, 2**n * 8),
                arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
plt.tight_layout()
plt.show()

### Animation: a single qubit sweeping superposition

A single qubit's state can be written as
$$|\psi\rangle = \cos(\theta/2)\,|0\rangle + e^{i\varphi}\sin(\theta/2)\,|1\rangle.$$

When θ=0 the qubit is just |0⟩; when θ=π it's |1⟩; in between, it's in a superposition. The animation below sweeps θ and shows the probability of measuring 0 vs 1.

Run the cell, the GIF will be embedded in the notebook output.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

theta_vals = np.linspace(0, 2*np.pi, 80)
bars = ax1.bar(['|0⟩', '|1⟩'], [1, 0], color=['#2A9D8F', '#E76F51'])
ax1.set_ylim(0, 1.05); ax1.set_ylabel("Probability"); ax1.set_title("Measurement probabilities")

circ = patches.Circle((0,0), 1, fill=False, color='gray')
ax2.add_patch(circ)
ax2.plot([-1,1],[0,0], color='lightgray', lw=0.5); ax2.plot([0,0],[-1,1], color='lightgray', lw=0.5)
ax2.set_xlim(-1.2,1.2); ax2.set_ylim(-1.2,1.2); ax2.set_aspect('equal')
ax2.set_title("Bloch-disk slice (φ=0)")
ax2.text(0, 1.07, "|0⟩", ha='center'); ax2.text(0, -1.15, "|1⟩", ha='center')
arrow, = ax2.plot([0,0],[0,1], color='#5B2C91', lw=3)

def update(i):
    theta = theta_vals[i]
    p0 = np.cos(theta/2)**2
    p1 = np.sin(theta/2)**2
    bars[0].set_height(p0); bars[1].set_height(p1)
    arrow.set_data([0, np.sin(theta)], [0, np.cos(theta)])
    return bars[0], bars[1], arrow

anim = FuncAnimation(fig, update, frames=len(theta_vals), interval=70, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

### A tiny simulator: superposition of 3 qubits

Let's actually build a state vector by hand. A 3-qubit register has 2³ = 8 basis states. The Hadamard gate H puts a single qubit into an equal superposition; applying H to every qubit gives the *uniform* superposition over all 8 states.

In [ ]:
def kron_all(matrices):
    """Tensor (Kronecker) product of a list of matrices."""
    out = matrices[0]
    for m in matrices[1:]:
        out = np.kron(out, m)
    return out

H = (1/np.sqrt(2)) * np.array([[1, 1],
                                [1,-1]])
I = np.eye(2)

# Start in |000>
n = 3
state = np.zeros(2**n); state[0] = 1.0

# Apply H to every qubit
H_all = kron_all([H]*n)
state = H_all @ state

# Probabilities of each basis state
probs = np.abs(state)**2
labels = [format(i, f'0{n}b') for i in range(2**n)]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(labels, probs, color='#5B2C91')
ax.set_ylim(0, 0.3)
ax.set_ylabel("Probability  |amplitude|²")
ax.set_xlabel("Basis state")
ax.set_title("After H⊗H⊗H on |000⟩, every state is equally likely")
plt.tight_layout(); plt.show()

print("State vector amplitudes:")
for lbl, amp in zip(labels, state):
    print(f"  |{lbl}⟩ : {amp:+.4f}")

### 💻 Activity 2.1, Hyperparameter Superposition

**Scenario:** You're tuning a neural network with 4 binary hyperparameters (e.g., batchnorm on/off, dropout on/off, augmentation on/off, big-vs-small model). That's 2⁴ = 16 configurations.

**Your task:** Build the uniform superposition over all 16 configurations, then measure 1,000 times. Plot a histogram of which configuration the measurements landed on.

Fill in the three `# TODO` lines. The cell below is intentionally incomplete, try it before peeking at the solution.

In [ ]:
# ---- Activity 2.1: complete the TODOs ----
n_qubits = 4

# TODO 1: prepare the initial state |0000> on n qubits
state = np.zeros(2**n_qubits)
state[0] = 1.0

# TODO 2: build the H-on-all-qubits operator
# Hint: use kron_all and H from earlier.
H_all = None   # <-- replace None

# TODO 3: apply H_all to state, then compute probs.
probs = None   # <-- replace None

# ---- sampling & plotting (already written) ----
if H_all is not None and probs is not None:
    samples = np.random.choice(2**n_qubits, size=1000, p=probs)
    counts = np.bincount(samples, minlength=2**n_qubits)
    cfgs = [format(i, f'0{n_qubits}b') for i in range(2**n_qubits)]
    plt.figure(figsize=(10,3))
    plt.bar(cfgs, counts, color='#2A9D8F')
    plt.xticks(rotation=45); plt.ylabel("Times sampled")
    plt.title("1,000 measurements of a uniform 4-qubit superposition")
    plt.tight_layout(); plt.show()
else:
    print("Fill in the TODOs above to see the plot.")

<details>
<summary>✅ Click for the Activity 2.1 solution</summary>

```python
H_all = kron_all([H]*n_qubits)
state = H_all @ state
probs = np.abs(state)**2
```

Each configuration should be sampled ~1000/16 ≈ 62 times. The bars will look roughly even, that's quantum parallelism in action.
</details>

---
# Section 2.2, Grover's Search and Optimization

## The search problem in AI

Suppose you have N items and exactly one is "good." Classically, you have to look at about N/2 items on average, and N in the worst case, to find it.

**Examples in AI:**
- Finding the one weight initialization (out of millions tried) that makes a model converge fastest.
- Searching a database of embeddings for the closest match to a query.
- Looking for a satisfying assignment to a SAT formula that encodes a planning problem.
- Sampling a discrete action from a learned policy that scores actions with an oracle.

**Grover's algorithm** solves this in roughly √N steps, a *quadratic* speed-up. For N = 1,000,000 items, classical takes ~500,000 lookups; Grover takes ~1,000.

## The main idea: amplitude amplification

Grover repeats two steps:

1. **Oracle**, flip the sign of the amplitude of the "good" state.
2. **Diffusion**, reflect every amplitude about the *average* amplitude.

The combined effect is to rotate the state vector closer and closer to the good state. After ≈ (π/4)·√N iterations, the good state has near-unit probability.

Below is a diagram of one iteration:

In [ ]:
# Static diagram of amplitude amplification on N=8 states.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
N = 8; target = 5
amps = np.ones(N) / np.sqrt(N)

# Step 0: uniform
axes[0].bar(range(N), amps, color=['#5B2C91' if i==target else '#bdbdbd' for i in range(N)])
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_title("(1) Uniform superposition")
axes[0].set_ylim(-0.7, 0.7)

# Step 1: oracle flips target sign
amps2 = amps.copy(); amps2[target] = -amps2[target]
axes[1].bar(range(N), amps2, color=['#E76F51' if i==target else '#bdbdbd' for i in range(N)])
axes[1].axhline(amps2.mean(), color='gray', ls='--', label=f'mean = {amps2.mean():.2f}')
axes[1].axhline(0, color='k', lw=0.5); axes[1].legend(loc='upper right', fontsize=8)
axes[1].set_title("(2) Oracle: flip target amplitude")
axes[1].set_ylim(-0.7, 0.7)

# Step 2: diffuse about the mean
mean = amps2.mean()
amps3 = 2*mean - amps2
axes[2].bar(range(N), amps3, color=['#2A9D8F' if i==target else '#bdbdbd' for i in range(N)])
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_title("(3) Diffusion: reflect about mean")
axes[2].set_ylim(-0.7, 0.7)

for ax in axes:
    ax.set_xlabel("Basis state"); ax.set_ylabel("Amplitude")
plt.suptitle("One Grover iteration on N=8 (target = state 5)", y=1.02)
plt.tight_layout(); plt.show()

### Animation: Grover iterations on a 16-item search

The animation runs Grover on N=16 items, target = item 11. Watch the green bar (the target) grow while the gray ones shrink. After about ⌊(π/4)·√16⌋ = 3 iterations, the probability of the target is near 1.

In [ ]:
def grover_step(amps, target):
    a = amps.copy()
    a[target] = -a[target]            # oracle
    m = a.mean()
    return 2*m - a                    # diffusion

N = 16; target = 11
amps = np.ones(N) / np.sqrt(N)
frames = [amps.copy()]
for _ in range(8):
    amps = grover_step(amps, target)
    frames.append(amps.copy())

fig, ax = plt.subplots(figsize=(9, 3.8))
colors = ['#2A9D8F' if i==target else '#bdbdbd' for i in range(N)]
bars = ax.bar(range(N), frames[0], color=colors)
ax.set_ylim(-1.05, 1.05); ax.set_xlabel("State"); ax.set_ylabel("Amplitude")
title = ax.set_title("Iteration 0")
ax.axhline(0, color='k', lw=0.5)

def update(i):
    for b, h in zip(bars, frames[i]):
        b.set_height(h)
    p = frames[i][target]**2
    title.set_text(f"Iteration {i}  |  P(target) = {p:.3f}")
    return bars

anim = FuncAnimation(fig, update, frames=len(frames), interval=900, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

### Code-along: a full Grover implementation (state-vector simulator)

We'll implement Grover entirely with NumPy. The same code works on any N that is a power of 2.

In [ ]:
def grovers_algorithm(n_qubits, marked_state, n_iterations=None):
    """Run Grover's algorithm on n_qubits searching for marked_state.

    Returns the final probability distribution over basis states.
    """
    N = 2**n_qubits
    if n_iterations is None:
        n_iterations = int(round(np.pi/4 * np.sqrt(N)))

    # 1. start uniform
    state = np.ones(N) / np.sqrt(N)

    history = [state.copy()]
    for _ in range(n_iterations):
        # Oracle: flip sign of marked state
        state[marked_state] *= -1
        # Diffusion: 2|s><s| - I, where |s> is uniform
        mean = state.mean()
        state = 2*mean - state
        history.append(state.copy())

    return state, history, n_iterations

# Search 64 items for item #42
final_state, history, n_iter = grovers_algorithm(n_qubits=6, marked_state=42)
probs = final_state**2

print(f"Iterations used: {n_iter}")
print(f"P(target=42) after Grover: {probs[42]:.4f}")
print(f"P(target=42) random guess: {1/64:.4f}")
print(f"Speed-up factor in probability: {probs[42] / (1/64):.1f}×")

plt.figure(figsize=(11,3))
plt.bar(range(64), probs, color=['#2A9D8F' if i==42 else '#cccccc' for i in range(64)])
plt.xlabel("State"); plt.ylabel("Probability")
plt.title("Final probability distribution after Grover (N=64, target=42)")
plt.tight_layout(); plt.show()

### 💻 Activity 2.2, Find the rogue model

**Scenario:** You have 32 trained models. Exactly one of them is *poisoned* (a backdoor was inserted during training). A black-box test can tell you "yes/no, this is the poisoned one" but you want to find which one with as few calls as possible.

**Your task:** Use the `grovers_algorithm` function to find the poisoned model. Then verify, by trying every possible `n_iterations` from 1 to 10, that the success probability *oscillates*, Grover doesn't keep improving forever. Plot that oscillation.

In [ ]:
# ---- Activity 2.2 ----
# Set up: 32 models, the poisoned one is at index 19.
poisoned = 19
n_qubits = 5     # because 2**5 = 32

# TODO 1: call grovers_algorithm on n_qubits, marked_state=poisoned, and default iterations.
# Save the return values into final_state, _, n_iter and print the peak probability.
final_state, _, n_iter = None, None, None    # <-- replace with your call

# TODO 2: sweep n_iterations from 1 to 10 and record P(poisoned) each time.
# Store the probabilities in a list called probs_over_iters.
probs_over_iters = None                       # <-- replace with your list

# ---- plotting (already written, only runs after you complete both TODOs) ----
if final_state is not None and probs_over_iters is not None:
    print(f"After {n_iter} optimal iterations: P(poisoned) = {final_state[poisoned]**2:.4f}")
    plt.figure(figsize=(8, 3.5))
    plt.plot(range(1, 11), probs_over_iters, marker='o', color='#5B2C91', lw=2)
    plt.axvline(n_iter, color='gray', ls='--', label=f'optimal ~ {n_iter}')
    plt.xlabel("Number of Grover iterations")
    plt.ylabel("P(find poisoned model)")
    plt.title("Too many iterations is worse than the right amount")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
else:
    print("Fill in the TODOs above to see the plot.")

<details>
<summary>🤔 Why does the curve come back down?</summary>

Each Grover iteration rotates the state by a fixed angle in a 2-D subspace. Doing *too many* rotations overshoots the target and rotates away from it. This is why estimating the number of marked items (or doing **amplitude amplification with unknown counts**) is its own subfield.
</details>

---
# Section 2.3, The Quantum Fourier Transform

## Classical Fourier transform, a refresher

The discrete Fourier transform (DFT) takes a length-N sequence of complex numbers $x_0, x_1, \ldots, x_{N-1}$ and produces

$$ X_k = \sum_{j=0}^{N-1} x_j \, e^{-2\pi i \, j k / N}, \qquad k = 0, 1, \ldots, N-1.$$

Intuitively, the DFT decomposes a signal into its frequency components, it tells you *which oscillation rates are present*.

**Where you've already seen it:**

| Domain | Use of FT |
|---|---|
| Audio / speech | Spectrograms, MFCCs for speech recognition |
| Computer vision | Frequency-domain filters, JPEG compression |
| Time-series ML | Detecting periodic structure |
| PDE-based scientific ML | Fourier neural operators |
| Convolutional networks | Convolution = pointwise multiplication in Fourier space |

A naïve DFT costs O(N²) operations. The famous **Fast Fourier Transform (FFT)** reduces this to O(N log N).

## Quantum Fourier Transform (QFT)

The QFT does the *same* linear transformation, but on the amplitudes of a quantum state:

$$ \text{QFT}\,|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i j k / N} \, |k\rangle.$$

Two things to notice:

1. The QFT runs in **O((log N)²)** quantum gates, *exponentially* faster than even the FFT.
2. The catch: you can't read all N transformed amplitudes, you can only sample one. So the QFT is most useful as a **subroutine** inside a bigger algorithm that knows what to do with that sample.

## Why AI cares

- **Shor's algorithm** uses QFT for period finding, relevant to AI security.
- **HHL** and quantum linear-algebra routines use it for solving linear systems that show up in regression and kernel methods.
- **Quantum signal processing** for time-series and sequence models.
- **Phase estimation** (which uses QFT) is the workhorse behind many quantum-chemistry and ML kernel routines.

### Classical FT in action

Let's build a signal of two sine waves plus noise, then recover the frequencies with the FFT.

In [ ]:
# Build a noisy 2-tone signal
fs = 256                                # sample rate
t = np.linspace(0, 1, fs, endpoint=False)
signal = (np.sin(2*np.pi*7*t)           # 7 Hz
        + 0.6*np.sin(2*np.pi*32*t)      # 32 Hz
        + 0.4*np.random.randn(fs))      # noise

spectrum = np.fft.fft(signal)
freqs = np.fft.fftfreq(fs, d=1/fs)
mask = freqs >= 0

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(t, signal, color='#5B2C91'); axes[0].set_title("Signal in the time domain")
axes[0].set_xlabel("t (s)"); axes[0].set_ylabel("amplitude")
axes[1].stem(freqs[mask], np.abs(spectrum[mask]), basefmt=" ", linefmt='#2A9D8F', markerfmt='o')
axes[1].set_xlim(0, 60); axes[1].set_title("Magnitude of the Fourier transform")
axes[1].set_xlabel("frequency (Hz)"); axes[1].set_ylabel("|X(f)|")
plt.tight_layout(); plt.show()
print("The two tall spikes mark the 7 Hz and 32 Hz components, exactly what we put in.")

### Building the QFT matrix by hand

The QFT on n qubits is an N×N matrix (N = 2ⁿ) whose entries are roots of unity.

In [ ]:
def qft_matrix(n_qubits):
    N = 2**n_qubits
    omega = np.exp(2j * np.pi / N)
    j, k = np.meshgrid(np.arange(N), np.arange(N), indexing='ij')
    return omega**(j*k) / np.sqrt(N)

QFT3 = qft_matrix(3)

# Visualize the QFT matrix: phases as colors
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(np.abs(QFT3), cmap='viridis')
axes[0].set_title("|QFT| , every entry has the same magnitude")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(np.angle(QFT3), cmap='twilight')
axes[1].set_title("arg(QFT) , phase is what does the work")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label='radians')
plt.tight_layout(); plt.show()

# Sanity: QFT is unitary, i.e. QFT^dagger @ QFT = I
err = np.linalg.norm(QFT3.conj().T @ QFT3 - np.eye(8))
print(f"Unitarity check: ||QFT^dagger · QFT - I|| = {err:.2e}  (should be ~0)")

### Applying the QFT to a quantum state

Watch what happens when we apply the QFT to a periodic state. The Fourier transform turns periodicity into a sharp spike, the same trick QFT is famous for.

In [ ]:
n = 4                       # 16 amplitudes
N = 2**n
period = 4                  # state is non-zero every 4 indices

state = np.zeros(N, dtype=complex)
state[::period] = 1
state /= np.linalg.norm(state)

QFT = qft_matrix(n)
transformed = QFT @ state

fig, axes = plt.subplots(1, 2, figsize=(11,3.5))
axes[0].bar(range(N), np.abs(state)**2, color='#5B2C91')
axes[0].set_title(f"Input: periodic state (period={period})")
axes[0].set_xlabel("basis state"); axes[0].set_ylabel("prob")

axes[1].bar(range(N), np.abs(transformed)**2, color='#2A9D8F')
axes[1].set_title("After QFT: sharp spikes at multiples of N/period")
axes[1].set_xlabel("basis state"); axes[1].set_ylabel("prob")
plt.tight_layout(); plt.show()

### 💻 Activity 2.3, QFT as a period detector

**Scenario:** A reinforcement-learning agent's reward signal seems to have a hidden periodicity. You suspect period = 8 in a length-32 trace, but the trace is noisy.

**Your task:** Build a length-32 binary signal with period 8 plus a little randomness, then use *both* the classical FFT and the QFT (matrix multiplication) to recover the period. Compare them.

In [ ]:
# ---- Activity 2.3 ----
# Goal: recover the period of a noisy periodic signal using the QFT.
N = 32
true_period = 8

# TODO 1: build a length-N binary signal that is 1 every `true_period` steps, 0 otherwise.
# Hint: [1.0 if i % true_period == 0 else 0.0 for i in range(N)]
trace = None                                # <-- replace

# TODO 2: add small Gaussian noise (std=0.1) and normalize.
noisy = None                                # <-- replace

# ---- QFT (already written, runs after your TODOs) ----
if trace is not None and noisy is not None:
    QFT = qft_matrix(int(np.log2(N)))
    qft_result = QFT @ noisy.astype(complex)
    qft_probs = np.abs(qft_result)**2

    fft_probs = np.abs(np.fft.fft(noisy))**2
    fft_probs /= fft_probs.sum()

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.3))
    axes[0].stem(range(N), noisy, basefmt=" "); axes[0].set_title("Noisy signal")
    axes[1].stem(range(N), fft_probs, basefmt=" ", linefmt='#5B2C91', markerfmt='o')
    axes[1].set_title("Classical FFT probabilities")
    axes[2].stem(range(N), qft_probs, basefmt=" ", linefmt='#2A9D8F', markerfmt='o')
    axes[2].set_title("QFT probabilities")
    for ax in axes: ax.set_xlabel("index")
    plt.tight_layout(); plt.show()

    # Robust period recovery: a period-P impulse train has equal-height peaks at
    # multiples of N/P. So the CORRECT way to infer the period is to look at the
    # spacing between peaks, not just argmax (which is ambiguous).
    threshold = 0.5 * qft_probs.max()
    peak_indices = np.where(qft_probs[1:N//2] > threshold)[0] + 1
    if len(peak_indices) >= 1:
        first_peak = int(peak_indices[0])
        inferred_period = N // first_peak
        print(f"Peaks above half-max (in first half spectrum): {list(peak_indices)}")
        print(f"First peak at index {first_peak}, so inferred period = N / peak = {N}/{first_peak} = {inferred_period}")
    else:
        print("No clear peak found. Try lowering noise.")
else:
    print("Fill in the TODOs above to see the plot.")

<details>
<summary>📌 Why doesn't QFT just replace the FFT?</summary>

Because measurement gives you **one sample**, not the whole spectrum. QFT shines when you only need to *sample* the dominant frequency, e.g., inside phase estimation. If your downstream task actually needs the entire spectrum (like building a spectrogram), classical FFT is fine.
</details>

---
# Section 2.4, Sampling and Probability Distributions

## Modern AI is sampling

Almost every probabilistic ML method, in the end, has to **sample from a complicated distribution**:

- **Bayesian inference**, drawing samples from the posterior P(θ | data).
- **Generative models**, VAEs, GANs, normalizing flows produce samples.
- **Diffusion models**, denoising = iterative sampling along a learned trajectory.
- **Monte Carlo methods**, MCMC, importance sampling, particle filters.
- **Reinforcement learning**, sampling actions, trajectories, advantages.

The catch is that *exact* sampling from high-dimensional, multi-modal distributions is hard. MCMC mixes slowly. Variational methods give biased samples. This is where quantum gets interesting.

## Quantum computers as sampling machines

Every measurement of a quantum state is, by construction, a draw from the distribution

$$ P(x) = |\langle x | \psi \rangle|^2.$$

If you can *prepare* a state whose amplitudes encode a distribution you care about, you get exact samples almost for free. This is the basis of:

- **Boltzmann machine training**, sampling from the Gibbs distribution is the bottleneck. D-Wave and quantum-annealing approaches target exactly this.
- **Born machines / quantum circuit Born machines (QCBM)**, quantum analogs of generative models.
- **Quantum-enhanced MCMC**, using a quantum walk to mix faster than classical Markov chains.

### Visualization: classical vs quantum sampling of a tricky distribution

Imagine a target distribution with two narrow modes far apart. Classical MCMC (here, a Metropolis sampler) gets *stuck* in one mode for a long time. A quantum sampler that directly encodes the target draws from both modes immediately.

In [ ]:
# Bimodal target: two narrow Gaussians
def target(x):
    return 0.5*np.exp(-0.5*((x+3)/0.3)**2) + 0.5*np.exp(-0.5*((x-3)/0.3)**2)

xs = np.linspace(-6, 6, 600)
tgt = target(xs); tgt /= tgt.sum()

# --- Naive MCMC ---
def naive_mcmc(n_steps, proposal_std=0.5, x0=-3.0):
    x = x0; trace = [x]
    for _ in range(n_steps-1):
        xp = x + np.random.randn() * proposal_std
        a = min(1.0, target(xp) / max(target(x), 1e-12))
        if np.random.rand() < a:
            x = xp
        trace.append(x)
    return np.array(trace)

mcmc_samples = naive_mcmc(3000)

# --- "Quantum" sampler: direct sample from target ---
quantum_samples = np.random.choice(xs, size=3000, p=tgt)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.3))
axes[0].plot(xs, tgt*len(xs), color='black'); axes[0].set_title("Target distribution"); axes[0].fill_between(xs, tgt*len(xs), alpha=0.2)

axes[1].hist(mcmc_samples, bins=60, density=True, color='#E76F51', alpha=0.8)
axes[1].plot(xs, target(xs)/np.trapz(target(xs), xs), color='black', lw=1)
axes[1].set_title("Classical MCMC after 3000 steps (mode-trapped)")
axes[1].set_xlim(-6, 6)

axes[2].hist(quantum_samples, bins=60, density=True, color='#2A9D8F', alpha=0.8)
axes[2].plot(xs, target(xs)/np.trapz(target(xs), xs), color='black', lw=1)
axes[2].set_title("Quantum-style direct sampling (no chain)")
axes[2].set_xlim(-6, 6)
plt.tight_layout(); plt.show()

### Building a quantum sampler from amplitudes

We can *encode* any non-negative distribution into the amplitudes of a quantum state. Here's a tiny construction.

In [ ]:
# Target: a discrete distribution over 16 outcomes (looks like a skewed bell)
N = 16
target_pmf = np.exp(-0.5 * ((np.arange(N) - 5)/2.2)**2)
target_pmf /= target_pmf.sum()

# Amplitudes that reproduce target_pmf
amps = np.sqrt(target_pmf)             # any phase works; pick zero
assert np.isclose(np.sum(amps**2), 1.0)

# "Measure" by sampling under |amp|^2
samples = np.random.choice(N, size=4000, p=amps**2)
counts = np.bincount(samples, minlength=N) / 4000

x = np.arange(N)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(x-0.18, target_pmf, width=0.35, label="target P(x)", color='#5B2C91')
ax.bar(x+0.18, counts,     width=0.35, label="4000 quantum measurements", color='#2A9D8F')
ax.set_xlabel("outcome"); ax.set_ylabel("probability")
ax.set_title("Encoding a distribution in amplitudes → direct sampling")
ax.legend(); plt.tight_layout(); plt.show()

### Animation: a quantum walk samples a graph

A **quantum walk** is the quantum version of a random walk. On a cycle graph, it spreads out *quadratically* faster than a classical random walk, which is why quantum walks underpin many quantum sampling and search speed-ups.

In [ ]:
# Compare classical vs quantum walk spreading on a line.
T = 60
N = 81          # positions -40..+40
center = N // 2

# Classical: each step, go left or right with p=0.5
prob_classical = np.zeros((T, N)); prob_classical[0, center] = 1.0
for t in range(1, T):
    prob_classical[t, 1:-1] = 0.5 * prob_classical[t-1, :-2] + 0.5 * prob_classical[t-1, 2:]
    prob_classical[t] /= prob_classical[t].sum()

# Quantum walk on a line (coined walk, Hadamard coin)
coin_H = (1/np.sqrt(2)) * np.array([[1, 1],[1, -1]])
state = np.zeros((N, 2), dtype=complex); state[center, 0] = 1/np.sqrt(2); state[center, 1] = 1j/np.sqrt(2)
prob_quantum = np.zeros((T, N)); prob_quantum[0] = np.abs(state[:,0])**2 + np.abs(state[:,1])**2
for t in range(1, T):
    # apply coin
    state = state @ coin_H.T
    # shift: |0> moves left, |1> moves right
    new_state = np.zeros_like(state)
    new_state[:-1, 0] = state[1:, 0]
    new_state[1:, 1] = state[:-1, 1]
    state = new_state
    prob_quantum[t] = np.abs(state[:,0])**2 + np.abs(state[:,1])**2

fig, (axc, axq) = plt.subplots(1, 2, figsize=(11, 3.6))
xs = np.arange(N) - center
line_c, = axc.plot(xs, prob_classical[0], color='#E76F51', lw=2)
line_q, = axq.plot(xs, prob_quantum[0], color='#2A9D8F', lw=2)
for ax, title in [(axc, "Classical random walk"), (axq, "Quantum walk")]:
    ax.set_xlim(-40, 40); ax.set_ylim(0, 0.25)
    ax.set_xlabel("position"); ax.set_ylabel("probability"); ax.set_title(title)

txt = fig.suptitle("Step 0")

def update(t):
    line_c.set_ydata(prob_classical[t])
    line_q.set_ydata(prob_quantum[t])
    txt.set_text(f"Step {t}   |   quantum spread is wider")
    return line_c, line_q

anim = FuncAnimation(fig, update, frames=T, interval=110)
plt.close(fig)
HTML(anim.to_jshtml())

### 💻 Activity 2.4, Build your own amplitude-encoded sampler

**Scenario:** You're training a generative model and want a discrete distribution over 8 classes that looks like the *softmax* of some logits.

**Your task:** Build the quantum state whose measurement distribution matches `softmax(logits)`. Then run 5,000 measurements and verify your histogram matches the softmax curve.

In [ ]:
# ---- Activity 2.4 ----
logits = np.array([0.2, 1.8, 0.5, 2.4, 0.1, 1.2, 3.0, 0.6])

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

# TODO 1: compute the target distribution from logits.
target = None                                # <-- replace with softmax(logits)

# TODO 2: build amplitudes whose squared moduli match target.
amps = None                                  # <-- replace with sqrt of target

# TODO 3: draw 5000 samples using np.random.choice with the correct p= argument.
samples = None                               # <-- replace

# ---- plotting (already written) ----
if target is not None and amps is not None and samples is not None:
    hist = np.bincount(samples, minlength=len(target)) / 5000
    x = np.arange(len(target))
    plt.figure(figsize=(8,3.3))
    plt.bar(x-0.18, target, width=0.35, label='softmax(logits)', color='#5B2C91')
    plt.bar(x+0.18, hist,   width=0.35, label='5000 measurements', color='#2A9D8F')
    plt.xlabel("class"); plt.ylabel("probability"); plt.legend()
    plt.title("Amplitude encoded sampler reproduces the target")
    plt.tight_layout(); plt.show()
else:
    print("Fill in the TODOs above to see the plot.")

---
## Unit 2 Summary

A quick recap of the concepts you saw in this unit:

| Concept | Key idea |
|---|---|
| **Superposition** | n qubits can hold 2ⁿ basis states at the same time |
| **Interference** | Algorithms shape amplitudes so the right answer becomes more likely |
| **Grover's algorithm** | Quadratic speedup for unstructured search via amplitude amplification |
| **Quantum Fourier Transform** | Exponentially fast transform, used as a subroutine inside bigger algorithms |
| **Measurement is sampling** | Every measurement draws a sample from \|amplitude\|² |
| **Quantum sampling** | Encode a distribution in amplitudes, then measure to sample exactly |
| **AI applications** | Unstructured search, period finding, generative modeling, MCMC |

In the next unit, we put these building blocks together into actual quantum machine-learning models.

---

---
# End-of-Unit Quiz

Ten multiple choice questions. Reveal the answer to check yourself.

**1. A quantum computer with n qubits can represent a superposition over how many basis states?**

A) n
B) n^2
C) 2^n
D) n!

<details><summary>Answer 1</summary>**C) 2^n.** Each qubit doubles the state space.</details>

---

**2. Which statement about quantum parallelism is correct?**

A) A measurement gives you every parallel result.
B) Parallelism alone guarantees a speed-up.
C) Quantum parallelism produces a superposition of all computations, but interference and measurement design extract a useful answer.
D) Quantum parallelism is identical to GPU parallelism.

<details><summary>Answer 2</summary>**C).** Without clever interference, you get a uniformly random output.</details>

---

**3. Grover's algorithm finds a marked item in an unsorted database of size N in roughly:**

A) O(log N) steps
B) O(sqrt(N)) steps
C) O(N) steps
D) O(N log N) steps

<details><summary>Answer 3</summary>**B) O(sqrt(N)).** A quadratic speed-up over classical.</details>

---

**4. The Quantum Fourier Transform requires how many gates on n qubits?**

A) O(n)
B) O(n^2)
C) O(2^n)
D) O(n * 2^n)

<details><summary>Answer 4</summary>**B) O(n^2).** Exponentially faster in gate count than the classical FFT.</details>

---

**5. Why are quantum computers natural sampling machines?**

A) Qubits flip randomly, so any measurement is a sample.
B) Measurement produces outcome x with probability |<x|psi>|^2, exactly the distribution AI cares about.
C) They always produce uniform random numbers.
D) They run a built-in MCMC chain.

<details><summary>Answer 5</summary>**B).** Measurement IS sampling from |amplitude|^2.</details>

---

**6. Grover's algorithm needs roughly how many iterations to maximize the probability of finding a marked item?**

A) N
B) (pi/4) * sqrt(N)
C) log N
D) sqrt(N) * log N

<details><summary>Answer 6</summary>**B) about (pi/4) * sqrt(N).** More iterations than that will overshoot and rotate away from the answer.</details>

---

**7. The classical fast version of the Fourier transform is called:**

A) Slow Fourier Transform
B) Fast Fourier Transform (FFT)
C) Discrete Cosine Transform
D) Laplace Transform

<details><summary>Answer 7</summary>**B) FFT.** It runs in O(N log N).</details>

---

**8. Applying the QFT to a periodic input state gives an output that is:**

A) Uniform
B) Sharply peaked at multiples of N / period
C) Always zero
D) Random

<details><summary>Answer 8</summary>**B).** The Fourier transform of a periodic function is sharp spikes at the frequency.</details>

---

**9. The "oracle" in Grover's algorithm is a subroutine that:**

A) Predicts the future
B) Marks the correct answer by flipping its amplitude sign
C) Copies the input
D) Measures all qubits

<details><summary>Answer 9</summary>**B).** The oracle flips the sign of the amplitude of the target state; the diffusion operator then amplifies it.</details>

---

**10. Which AI technique is essentially a sampling problem at its core?**

A) Sorting numbers
B) Bayesian inference (drawing samples from a posterior)
C) Matrix multiplication
D) Linear regression by least squares

<details><summary>Answer 10</summary>**B) Bayesian inference.** So is MCMC, so are diffusion models and Boltzmann machines.</details>

---

## Unit 2 Summary

| Concept | Key idea |
|---|---|
| Superposition | n qubits hold 2^n basis states at once. |
| Interference | Algorithms shape amplitudes so the right answer becomes likely. |
| Grover's algorithm | Quadratic speed-up for unstructured search. |
| Quantum Fourier Transform | Exponentially fast transform, used as a subroutine. |
| Measurement = sampling | Every measurement draws from |amplitude|^2. |
| Quantum sampling | Encode a distribution in amplitudes, then measure. |
| AI applications | Search, period finding, generative modeling, MCMC. |

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
# Set CONFIRM to True after you have filled in your answers, then run this cell.
CONFIRM = False

quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_filled  = [v.strip().upper() for v in quiz_answers.values() if v.strip()]
_valid   = all(v in {"A","B","C","D"} for v in _filled)
_all_ten = len(_filled) == 10

if not CONFIRM:
    print("Answers NOT submitted yet.")
    print("Fill in your ten answers above, then set CONFIRM = True and re-run this cell.")
elif not _all_ten:
    print(f"Only {len(_filled)} of 10 answers are filled in.")
    print("Fill in all 10, or set CONFIRM = 'PARTIAL' to force submit.")
elif not _valid:
    print("At least one answer is not A, B, C, or D. Please fix and re-run.")
else:
    _post_event("unit_completed",
                payload={"quiz": quiz_answers, "reflection": reflection})
    print("Submitted Unit 2! All 10 answers logged.")

if CONFIRM == "PARTIAL":
    _post_event("unit_completed",
                payload={"quiz": quiz_answers, "reflection": reflection, "partial": True})
    print("Partial submission logged.")